# DiReCT thickness engine — tensor interface

Compare [the antsimage interface](direct_cortical_thickness_antsimage.ipynb).
Both use the real `r16` tutorial image, the same estimated tissue maps, and
identical DiReCT settings. The image and tensor functions use ANTsTorch's engine;
`antstorch.direct.kelly_kapowski` is distinct from ANTsPy's `ants.kelly_kapowski`.


The inputs are an acquired MRI slice and intensity-based tissue estimates from
ANTsPy Atropos. No simulated anatomy or handcrafted tissue phantom is used.
This **2-D interface demonstration is not a volumetric cortical thickness
measurement**. The intensity clusters are illustrative tissue estimates, not
manually validated GM/WM labels. For 3-D analysis with Deep Atropos, use
[cortical_thickness.ipynb](cortical_thickness.ipynb) and supply the full-volume
segmentation and probability images to the chosen DiReCT interface. These short
iteration counts demonstrate the call, not convergence or anatomical accuracy.

Install ANTsTorch and notebook tools from the repository root before opening this notebook:
```bash
python -m pip install -e '.[notebooks]'
python -m ipykernel install --user --name python3 --display-name "Python 3 (ANTsTorch)"
```
Select that kernel and run the cells in order. All examples default to CPU.

In [ ]:
from time import perf_counter
_example_started = perf_counter()

import torch  # Import before ANTs to avoid OpenMP conflicts on macOS.
import numpy as np
import ants
import antstorch

# Display figures inside Jupyter/VS Code even if MPLBACKEND was set to Agg.
from IPython import get_ipython
shell = get_ipython()
if shell is not None:
    shell.run_line_magic("matplotlib", "inline")
import matplotlib.pyplot as plt

device = "cpu"
antstorch.set_default_device(device)

# Synchronize the selected accelerator so timings include completed GPU work.
from contextlib import contextmanager
_example_operation_times = []

def _synchronize_example_device():
    selected = torch.device(device)
    if selected.type == "cuda":
        torch.cuda.synchronize(selected)
    elif selected.type == "mps":
        torch.mps.synchronize()

@contextmanager
def time_step(label):
    _synchronize_example_device()
    started = perf_counter()
    try:
        yield
    finally:
        _synchronize_example_device()
        seconds = perf_counter() - started
        _example_operation_times.append((label, seconds))
        print(f"{label}: {seconds:.4f} s")

In [ ]:
# Estimate tissues from the real tutorial image; no synthetic segmentation or weights.
reference = ants.image_read(ants.get_ants_data("r16")).clone("float")
reference = ants.resample_image(reference, (96, 96), use_voxels=True)
foreground = ants.get_mask(reference)
with time_step("Atropos tissue preparation — CPU"):
    tissues = ants.atropos(a=reference, x=foreground, i="KMeans[3]",
                           m="[0.2,1x1]", c="[5,0]")
labels = tissues["segmentation"].numpy()
# Assign low/middle/high intensity clusters to CSF/GM/WM for this interface demo.
means = [reference.numpy()[labels == label].mean() for label in (1, 2, 3)]
order = np.argsort(means) + 1
segmentation = reference * 0
segmentation[labels == order[1]] = 2
segmentation[labels == order[2]] = 3
gray = tissues["probabilityimages"][int(order[1])-1]
white = tissues["probabilityimages"][int(order[2])-1]
assert (segmentation.numpy() == 2).any() and (segmentation.numpy() == 3).any()
for image in (segmentation, gray, white):
    assert ants.image_physical_space_consistency(image, reference)
    assert np.isfinite(image.numpy()).all()
options = dict(iterations=5, integration_points=3, gradient_step=0.025,
               smoothing_sigma=1., velocity_smoothing_variance=1.5,
               optimizer="direct", regularizer="gaussian")

## Convert labels and probabilities without intensity normalization
The tensor engine takes `(1, 1, y, x)` scalar tensors plus an `ImageDomain` in
physical `(x, y)` order. Label values must remain 2 and 3, and probabilities
must retain their original scale. This r16 example uses an identity direction,
the same local-axis convention used internally by the ANTsImage DiReCT bridge.

In [ ]:
from antstorch.bspline_flows import ImageDomain
from antstorch.direct import direct_cortical_thickness
from antstorch.syn.bridge import ants_image_to_tensor, tensor_to_ants_image

np.testing.assert_allclose(reference.direction, np.eye(2))
domain = ImageDomain(reference.shape, spacing=reference.spacing, origin=reference.origin,
                     direction=tuple(map(tuple, reference.direction)))
seg_tensor = ants_image_to_tensor(segmentation, device, normalize=False)
gray_tensor = ants_image_to_tensor(gray, device, normalize=False)
white_tensor = ants_image_to_tensor(white, device, normalize=False)
with time_step("DiReCT — tensor"):
    result = direct_cortical_thickness(seg_tensor, gray_tensor, white_tensor, domain, **options)
thickness_tensor = result.thickness
assert thickness_tensor.shape == seg_tensor.shape
assert thickness_tensor.device == gray_tensor.device
assert torch.isfinite(thickness_tensor).all() and (thickness_tensor >= 0).all()
assert len(result.energy_history) == options["iterations"]
thickness = tensor_to_ants_image(thickness_tensor, reference)
assert ants.image_physical_space_consistency(thickness, reference)
print("Energy history:", result.energy_history)

In [ ]:
ants.plot(reference, overlay=segmentation, axis=2, overlay_cmap="jet", overlay_alpha=0.5)
ants.plot(reference, overlay=thickness, axis=2, overlay_cmap="Reds", overlay_alpha=0.6)
print("Thickness range in input physical units:", float(thickness.min()), float(thickness.max()))

The image interface returns an ANTsImage; the tensor interface returns a
`DiReCTResult` containing thickness, deformation tensors and energy history.
The 2-D output illustrates the API and should not be interpreted as a 3-D
cortical biomarker. The original ANTs implementation may differ numerically;
see the [DiReCT implementation guide](https://github.com/ANTsX/ANTsTorch/blob/main/antstorch/direct/README.md).

## Execution time

Run **Restart Kernel and Run All Cells** for a comparable end-to-end duration.
The total starts at the first setup cell and includes imports, downloads, data
preparation, computation, plotting and file writes. Kernel startup is excluded;
manual pauses between cells are included. Operation timings include any
preprocessing or asset downloads performed inside the called function. First
runs can be slower because caches are empty. Timings describe this run on the
selected device, not a fixed performance estimate for other machines.

In [ ]:
_synchronize_example_device()
total_seconds = perf_counter() - _example_started
operation_seconds = sum(seconds for _, seconds in _example_operation_times)
print(f"Total elapsed time: {total_seconds:.2f} s ({total_seconds / 60:.2f} min)")
print(f"Timed operations: {operation_seconds:.4f} s on {device}")